# Gait PD vs HC Pipeline — v3

**Researcher:** Joseph Daniel Buhagiar  
**Supervisor:** Dr. Clifford DeRaffaele, MCAST, Malta

**Research Questions**
- **RQ1 Feature Discriminability** — which features significantly separate PD from HC?
- **RQ2 Condition Sensitivity** — which walking condition is most effective?
- **RQ3 Model Validation** — how do RBF-SVM, Random Forest, and MLP perform under LOSO cross-validation?


In [ ]:

import os, glob, time, warnings, itertools
from pathlib import Path
from math import factorial

import numpy as np
import pandas as pd

from scipy.stats import (skew, kurtosis, mannwhitneyu,
                          friedmanchisquare, wilcoxon)
from scipy.signal import welch
from statsmodels.stats.multitest import multipletests

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

## 1. Configuration

Global constants: paths, window parameters, cohort balancing, frequency bands, model colours.


In [ ]:

DATASET_DIR = r"E:\DATASETT\wearGaitPDv2Data"
OUTPUT_DIR  = r"E:\DATASETT\results_pdvshc_v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

FS          = 100.0
WINDOW_SIZE = 200       # 2 s
STEP_SIZE   = 100       # 50% overlap

BALANCE_SUBJECTS       = True
BALANCE_N_PER_GROUP    = 40
BALANCE_SELECTION_SEED = RANDOM_SEED

CONDITIONS = ["SelfPace", "TUG", "TandemGait", "HurriedPace"]

FOLDER_GROUP_MAP = {
    "control participants":  0,
    "controll participants": 0,
    "pd participants":       1,
}
GROUP_LABELS = {0: "Healthy Control", 1: "Parkinson's Disease"}
GROUP_SHORT  = {0: "HC", 1: "PD"}
GROUP_PALETTE = {0: "steelblue", 1: "darkorange"}

KNOWN_PREFIXES = {
    "HC", "WHC", "NLS", "PD", "WPD",
    "NLSCONTROL", "NLSPD", "NLSHC", "MANIFEST",
}
PREFIX_DIAGNOSTIC_HINTS = {"HC": 0, "WHC": 0, "PD": 1, "WPD": 1}

MODEL_ORDER  = ["RBF_SVM", "RandomForest", "MLP"]
MODEL_COLORS = {
    "RBF_SVM": "royalblue",
    "RandomForest": "forestgreen",
    "MLP": "tomato",
}

# Frequency bands relevant to gait and PD tremor (Hz)
FREQ_BANDS = {
    "slow":    (0.1,  0.5),   # postural sway / slow drift
    "gait":    (0.5,  3.0),   # primary step/stride frequency
    "harmonic":(3.0,  8.0),   # gait harmonics + PD tremor (4-6 Hz)
    "highfreq":(8.0, 20.0),   # impact transients
}

# FDR significance threshold
ALPHA = 0.05

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

print("Configuration complete.")
print(f"  Dataset   : {DATASET_DIR}")
print(f"  Output    : {OUTPUT_DIR}")
print(f"  Window    : {WINDOW_SIZE} samples = {WINDOW_SIZE/FS:.1f}s | step={STEP_SIZE}")
print(f"  Balance   : {BALANCE_N_PER_GROUP} subjects/group")
print(f"  Freq bands: {FREQ_BANDS}")

## 2. File Discovery & Inventory

Scan dataset directory, assign folder-derived labels, balance cohort to 40 subjects/group, build lookup maps.


In [ ]:

def label_from_path(fp):
    parts = [p.lower() for p in Path(fp).parts]
    for key, lbl in FOLDER_GROUP_MAP.items():
        if key in parts:
            return lbl
    return None

def subject_id_raw(fp):
    return os.path.basename(fp).split("_")[0]

def task_from_path(fp):
    stem = os.path.splitext(os.path.basename(fp))[0]
    for tok in stem.split("_")[1:]:
        for c in CONDITIONS:
            if tok.lower() == c.lower():
                return c
    return None

def prefix_of(sid):
    return "".join(ch for ch in str(sid) if ch.isalpha()).upper()

def is_data_file(fp):
    base = os.path.basename(fp).lower()
    stem = os.path.splitext(base)[0]
    return not base.startswith("manifest") and "_mat" not in stem

print("\n" + "="*68)
print("FILE DISCOVERY")
print("="*68)

all_csv  = sorted(glob.glob(os.path.join(DATASET_DIR,"**","*.csv"), recursive=True))
data_csv = sorted(f for f in all_csv if is_data_file(f))
print(f"Total CSVs: {len(all_csv)}  |  data CSVs after filter: {len(data_csv)}")

rows = []
for fp in data_csv:
    raw  = subject_id_raw(fp)
    pfx  = prefix_of(raw)
    lbl  = label_from_path(fp)
    task = task_from_path(fp)
    cf   = next((p for p in Path(fp).parts if p.lower() in FOLDER_GROUP_MAP), "")
    rows.append(dict(file=fp, raw_subject=raw, prefix=pfx,
                     label=lbl, task=task, class_folder=cf))

inv_raw = pd.DataFrame(rows)

unknown_pfx = [p for p in inv_raw["prefix"].unique() if p not in KNOWN_PREFIXES]
if unknown_pfx:
    print(f"INFO: unrecognised prefixes (label from folder): {unknown_pfx}")

no_label = inv_raw[inv_raw["label"].isna()]
if len(no_label):
    print(f"WARNING: {len(no_label)} files with no class folder — skipped.")

inv_raw = inv_raw[inv_raw["label"].notna()].copy()
inv_raw["label"] = inv_raw["label"].astype(int)
inv_raw["subject"] = inv_raw.apply(
    lambda r: f"{GROUP_SHORT[r['label']]}_{r['raw_subject']}", axis=1)

inv_cond = inv_raw[inv_raw["task"].isin(CONDITIONS)].copy().reset_index(drop=True)
print(f"Files with recognised class folder & task: {len(inv_cond)}")

rng = np.random.default_rng(BALANCE_SELECTION_SEED)
selected_by_group = {}
for lbl in sorted(inv_cond["label"].unique()):
    subs = np.array(sorted(inv_cond.loc[inv_cond["label"]==lbl,"subject"].unique()))
    n = min(BALANCE_N_PER_GROUP, len(subs))
    chosen = sorted(rng.choice(subs, size=n, replace=False).tolist())
    selected_by_group[int(lbl)] = chosen
    print(f"  {GROUP_LABELS[lbl]}: {len(subs)} available  →  {n} selected")

all_selected = set(s for v in selected_by_group.values() for s in v)
inventory = (inv_cond[inv_cond["subject"].isin(all_selected)]
             .sort_values(["label","subject","task"]).reset_index(drop=True))
print(f"Final inventory: {len(inventory)} trial files | "
      f"{inventory['subject'].nunique()} subjects")

FILE_LABEL_MAP   = dict(zip(inventory["file"], inventory["label"]))
FILE_SUBJECT_MAP = dict(zip(inventory["file"], inventory["subject"]))
selected_files   = inventory["file"].tolist()

## 3. Preprocessing

Select insole IMU channels (Acc/Gyr only), interpolate missing values, compute vector magnitudes.


In [ ]:

def get_insole_cols(df):
    cols = []
    for c in df.columns:
        lc = c.lower().replace("_","").replace(":","")
        if any(kw in lc for kw in ["freeacc","pressure","fsr","force"]):
            continue
        if any(lc.startswith(k) for k in
               ["linsoleacc","linsolegyr","rinsoleacc","rinsolegyr"]):
            cols.append(c)
        elif any(k in c.lower() for k in
                 ["linsole:acc","linsole:gyr","rinsole:acc","rinsole:gyr"]):
            cols.append(c)
    return cols

def add_magnitudes(data, sensor_cols):
    groups = {}
    for col in sensor_cols:
        if not any(col.lower().endswith(s) for s in ("_x","_y","_z")):
            continue
        base = col.rsplit("_",1)[0]
        groups.setdefault(base,[]).append(col)
    mag_cols = []
    for base, cols in groups.items():
        if len(cols) >= 2:
            name = f"{base}_Mag"
            data[name] = np.sqrt(sum(data[c].astype(float)**2 for c in cols))
            mag_cols.append(name)
    return data, mag_cols

def preprocess(df):
    sc = get_insole_cols(df)
    if not sc:
        return None, []
    data = df[sc].copy()
    for c in data.columns:
        data[c] = pd.to_numeric(data[c], errors="coerce")
    data.replace([np.inf,-np.inf], np.nan, inplace=True)
    data = data.interpolate(method="linear", limit_direction="both", axis=0)
    data = data.fillna(data.mean()).fillna(0)
    data, mc = add_magnitudes(data, sc)
    return data, sc + mc

## 4. Comprehensive Feature Extraction

21 features per channel:
- **Time domain (12):** mean, SD, CV, min, max, range, RMS, energy, skewness, kurtosis, ZCR, MCR
- **Frequency domain (8):** dominant freq, spectral entropy, total power, spectral centroid, PSD (slow/gait/harmonic/highfreq)
- **Entropy (1):** permutation entropy (order=3, normalised)


In [ ]:

def _safe(x):
    x = np.asarray(x, dtype=float)
    return x[np.isfinite(x)]


def _zcr(x):
    """Zero-crossing rate."""
    if len(x) < 2:
        return 0.0
    return float(np.sum(np.diff(np.sign(x)) != 0) / (len(x) - 1))


def _mcr(x):
    """Mean-crossing rate."""
    if len(x) < 2:
        return 0.0
    m = x - np.mean(x)
    return float(np.sum(np.diff(np.sign(m)) != 0) / (len(m) - 1))


def _perm_entropy(x, order=3):
    """Permutation entropy (normalised, bits). O(N*order)."""
    N = len(x)
    n = N - order + 1
    if n <= 0:
        return 0.0
    counts = {}
    for i in range(n):
        p = tuple(np.argsort(x[i:i+order]).tolist())
        counts[p] = counts.get(p, 0) + 1
    probs = np.array(list(counts.values()), dtype=float) / n
    h = -np.sum(probs * np.log2(probs + 1e-12))
    return float(h / np.log2(factorial(order)))   # normalise to [0,1]


def _psd(x, fs):
    """Return (frequencies, power) via Welch."""
    n = len(x)
    if n < 8:
        return np.zeros(1), np.zeros(1)
    f, P = welch(x, fs=fs, nperseg=min(64, n))
    return f, P


def time_features(x):
    x = _safe(x)
    if x.size == 0:
        return {k: 0.0 for k in [
            "mean","std","cv","min","max","range","rms",
            "energy","skewness","kurtosis","zcr","mcr"]}
    sd   = float(np.std(x))
    mn   = float(np.mean(x))
    cv   = float(sd / (abs(mn) + 1e-8))
    return {
        "mean":     mn,
        "std":      sd,
        "cv":       cv,
        "min":      float(np.min(x)),
        "max":      float(np.max(x)),
        "range":    float(np.max(x) - np.min(x)),
        "rms":      float(np.sqrt(np.mean(x**2))),
        "energy":   float(np.sum(x**2) / x.size),
        "skewness": float(skew(x))     if sd > 1e-8 else 0.0,
        "kurtosis": float(kurtosis(x)) if sd > 1e-8 else 0.0,
        "zcr":      _zcr(x),
        "mcr":      _mcr(x),
    }


def freq_features(x, fs=100.0):
    x = _safe(x)
    zeros = {"dominant_freq":0.0,"spectral_entropy":0.0,
             "total_power":0.0,"spectral_centroid":0.0,
             "psd_slow":0.0,"psd_gait":0.0,
             "psd_harmonic":0.0,"psd_highfreq":0.0}
    if x.size < 8:
        return zeros
    f, P = _psd(x, fs)
    total = P.sum()
    if total <= 0:
        return zeros
    p_norm = P / total

    feat = {}
    feat["dominant_freq"]    = float(f[np.argmax(P)])
    feat["spectral_entropy"] = float(-np.sum(p_norm * np.log(p_norm + 1e-12)))
    feat["total_power"]      = float(total)
    feat["spectral_centroid"]= float(np.sum(f * P) / total)

    for band_name, (flo, fhi) in FREQ_BANDS.items():
        mask = (f >= flo) & (f < fhi)
        bp = float(np.trapz(P[mask], f[mask])) if mask.sum() > 1 else 0.0
        feat[f"psd_{band_name}"] = bp

    return feat


def entropy_features(x):
    x = _safe(x)
    if x.size < 6:
        return {"perm_entropy": 0.0}
    return {"perm_entropy": _perm_entropy(x, order=3)}


def extract_all(window_df, fs=100.0):
    """Extract all features from one window (all channels)."""
    feat = {}
    for col in window_df.columns:
        x = window_df[col].values
        for k, v in time_features(x).items():
            feat[f"{col}__{k}"] = v
        for k, v in freq_features(x, fs).items():
            feat[f"{col}__{k}"] = v
        for k, v in entropy_features(x).items():
            feat[f"{col}__{k}"] = v
    return feat

# Feature descriptor count per channel
_N_FEAT_PER_CH = 12 + 8 + 1   # 21

## 5. Dataset Building

Apply sliding window (2 s, 50% overlap), aggregate to trial level (5 statistics per window feature), compute subject × condition means for statistical tests.


In [ ]:

def process_file(fp):
    label      = FILE_LABEL_MAP.get(fp)
    subject_id = FILE_SUBJECT_MAP.get(fp)
    task       = task_from_path(fp)
    if label is None:
        return [], [], [], [], []
    try:
        df = pd.read_csv(fp, low_memory=False)
    except Exception:
        return [], [], [], [], []
    data, all_cols = preprocess(df)
    if data is None or len(data) < WINDOW_SIZE:
        return [], [], [], [], []
    tid = f"{subject_id}_{task}"
    rows, labels, groups, trials, tasks = [], [], [], [], []
    for start in range(0, len(data) - WINDOW_SIZE + 1, STEP_SIZE):
        win = data[all_cols].iloc[start:start + WINDOW_SIZE]
        rows.append(extract_all(win, fs=FS))
        labels.append(label);  groups.append(subject_id)
        trials.append(tid);    tasks.append(task)
    return rows, labels, groups, trials, tasks


def build_dataset():
    all_rows, all_labels, all_groups, all_trials, all_tasks = [], [], [], [], []
    n  = len(selected_files)
    t0 = time.time()
    for i, fp in enumerate(selected_files, 1):
        if i % 50 == 0 or i == n:
            print(f"  [{i}/{n}] {time.time()-t0:.0f}s", end="\r")
        r, lb, g, t, tk = process_file(fp)
        all_rows.extend(r);  all_labels.extend(lb)
        all_groups.extend(g); all_trials.extend(t)
        all_tasks.extend(tk)
    print(f"\n  {len(all_rows)} windows extracted in {time.time()-t0:.1f}s")
    return (pd.DataFrame(all_rows), np.array(all_labels),
            np.array(all_groups), np.array(all_trials),
            np.array(all_tasks))


def aggregate_to_trial(X, y_arr, grp_arr, trial_arr, task_arr):
    meta = pd.DataFrame({"trial": trial_arr, "subject": grp_arr,
                          "task": task_arr, "label": y_arr})
    t_rows, t_labels, t_groups, t_ids, t_tasks = [], [], [], [], []
    for tid, idx in meta.groupby("trial").groups.items():
        idx = list(idx)
        Xg  = X.iloc[idx]
        feat = {}
        for col in X.columns:
            for suf, val in [
                ("trial_mean",   Xg[col].mean()),
                ("trial_std",    Xg[col].std() if len(Xg) > 1 else 0.0),
                ("trial_median", Xg[col].median()),
                ("trial_min",    Xg[col].min()),
                ("trial_max",    Xg[col].max()),
            ]:
                feat[f"{col}__{suf}"] = float(val) if np.isfinite(val) else 0.0
        t_rows.append(feat)
        g = meta.loc[idx]
        t_labels.append(int(g["label"].mode().iloc[0]))
        t_groups.append(str(g["subject"].mode().iloc[0]))
        t_ids.append(str(tid))
        t_tasks.append(str(g["task"].mode().iloc[0]))
    return (pd.DataFrame(t_rows), np.array(t_labels),
            np.array(t_groups), np.array(t_ids),
            np.array(t_tasks))


def subject_condition_means(X_win, y_win, grp_win, tsk_win):
    """Average window features to subject × condition level for statistics."""
    meta = pd.DataFrame({"subject": grp_win, "label": y_win, "task": tsk_win},
                         index=X_win.index)
    combined = pd.concat([meta, X_win], axis=1)
    return (combined.groupby(["subject","label","task"])[X_win.columns.tolist()]
            .mean().reset_index())


print("\nBuilding window-level feature matrix...")
X_win, y_win, grp_win, tri_win, tsk_win = build_dataset()
feat_cols = X_win.columns.tolist()
print(f"  Shape: {X_win.shape}")
print(f"  Features per channel: {_N_FEAT_PER_CH}  |  "
      f"Total window features: {len(feat_cols)}")
print(f"  Class balance: "
      f"{dict(zip(*np.unique(y_win, return_counts=True)))}")

print("\nAggregating to trial level...")
X_tr, y_tr, grp_tr, tri_tr, tsk_tr = aggregate_to_trial(
    X_win, y_win, grp_win, tri_win, tsk_win)
print(f"  Trial-level shape: {X_tr.shape}")
print(f"  Class balance: {dict(zip(*np.unique(y_tr, return_counts=True)))}")

print("\nBuilding subject × condition means for statistical analysis...")
subj_cond = subject_condition_means(X_win, y_win, grp_win, tsk_win)
print(f"  Subject-condition rows: {len(subj_cond)}")

# Per-condition trial slices (for LOSO)
X_cond, y_cond, grp_cond, tri_cond = {}, {}, {}, {}
for cond in CONDITIONS:
    m = tsk_tr == cond
    X_cond[cond]   = X_tr.loc[m].reset_index(drop=True)
    y_cond[cond]   = y_tr[m]
    grp_cond[cond] = grp_tr[m]
    tri_cond[cond] = tri_tr[m]
    hc  = int((y_tr[m]==0).sum())
    pdx = int((y_tr[m]==1).sum())
    print(f"  {cond:12s}: {m.sum():4d} trials  HC={hc}  PD={pdx}")

## 6. RQ1 — Feature Discriminability

Mann-Whitney U test for every feature × condition, FDR correction (Benjamini-Hochberg), rank-biserial correlation effect size.  
**Outputs:** volcano plots, top-features bar chart, group comparison plots.


In [ ]:

print("\n" + "="*68)
print("RQ1 — FEATURE DISCRIMINABILITY (Mann-Whitney U, FDR, Effect Size)")
print("="*68)

def rank_biserial(U, n1, n2):
    """Rank-biserial correlation as effect size for Mann-Whitney U."""
    return float(1.0 - 2.0 * U / (n1 * n2)) if n1 > 0 and n2 > 0 else np.nan

def run_mannwhitney_all_conditions(subj_cond_df, feat_cols):
    """
    For each feature × condition run Mann-Whitney U (PD vs HC).
    Returns a DataFrame with one row per (condition, feature).
    """
    records = []
    all_conditions = CONDITIONS + ["Pooled"]

    for cond in all_conditions:
        if cond == "Pooled":
            sub = subj_cond_df
        else:
            sub = subj_cond_df[subj_cond_df["task"] == cond]

        hc = sub[sub["label"] == 0]
        pd_ = sub[sub["label"] == 1]

        for feat in feat_cols:
            if feat not in sub.columns:
                continue
            hv = hc[feat].dropna().values
            pv = pd_[feat].dropna().values
            if len(hv) < 3 or len(pv) < 3:
                continue
            try:
                U, p = mannwhitneyu(pv, hv, alternative="two-sided")
            except Exception:
                U, p = np.nan, np.nan
            r = rank_biserial(U, len(pv), len(hv))
            records.append({
                "condition":   cond,
                "feature":     feat,
                "HC_mean":     float(np.mean(hv)),
                "HC_std":      float(np.std(hv)),
                "PD_mean":     float(np.mean(pv)),
                "PD_std":      float(np.std(pv)),
                "HC_n":        len(hv),
                "PD_n":        len(pv),
                "U":           float(U) if np.isfinite(U) else np.nan,
                "p_raw":       float(p) if np.isfinite(p) else np.nan,
                "effect_r":    r,
            })
    return pd.DataFrame(records)

stats_all = run_mannwhitney_all_conditions(subj_cond, feat_cols)

# Apply FDR correction (Benjamini-Hochberg) per condition
fdr_rows = []
for cond, grp in stats_all.groupby("condition"):
    valid = grp["p_raw"].notna()
    p_vals = grp.loc[valid, "p_raw"].values
    if len(p_vals) == 0:
        fdr_rows.append(grp)
        continue
    reject, p_fdr, _, _ = multipletests(p_vals, alpha=ALPHA, method="fdr_bh")
    grp = grp.copy()
    grp.loc[valid, "p_fdr"] = p_fdr
    grp.loc[valid, "significant"] = reject
    fdr_rows.append(grp)

stats_all = pd.concat(fdr_rows, ignore_index=True)
stats_all["p_fdr"]      = stats_all.get("p_fdr", np.nan)
stats_all["significant"]= stats_all.get("significant", False).fillna(False)
stats_all = stats_all.sort_values(["condition","p_raw"]).reset_index(drop=True)

# Save full stats table
stats_csv = os.path.join(OUTPUT_DIR, "SPSS_rq1_feature_stats_all.csv")
stats_all.to_csv(stats_csv, index=False)
print(f"  Full statistics table: {len(stats_all)} rows → {stats_csv}")

sig_stats = stats_all[stats_all["significant"] == True].copy()
print(f"\n  Significant features (FDR-corrected p < {ALPHA}):")
for cond, grp in sig_stats.groupby("condition"):
    print(f"    {cond:12s}: {len(grp):4d} significant features")

# ---- Top features across all conditions (by |effect_r|, FDR-significant) ----
top_feats_rq1 = (stats_all[stats_all["significant"]]
    .assign(abs_r=lambda d: d["effect_r"].abs())
    .sort_values("abs_r", ascending=False)
    .drop_duplicates("feature")
    .head(20))

print(f"\n  Top 20 globally discriminative features (FDR-sig, ranked by |r|):")
print(top_feats_rq1[["feature","condition","p_raw","p_fdr","effect_r"]].to_string())

top_feats_csv = os.path.join(OUTPUT_DIR, "SPSS_rq1_top_discriminative_features.csv")
top_feats_rq1.to_csv(top_feats_csv, index=False)

# ---- Figure 1: Volcano plot (per condition, one subplot each) ----
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5), sharey=False)
fig.suptitle("RQ1 — Volcano Plot: Feature Discriminability PD vs HC\n"
             "(x = effect size r, y = −log₁₀(p_raw), "
             "red = FDR-significant)", fontweight="bold", fontsize=12)

for ax, cond in zip(axes, CONDITIONS):
    cdata = stats_all[stats_all["condition"] == cond].dropna(subset=["p_raw","effect_r"])
    ns  = cdata[~cdata["significant"]]
    sig = cdata[cdata["significant"]]
    ax.scatter(ns["effect_r"],  -np.log10(ns["p_raw"]+1e-10),
               color="grey", alpha=0.3, s=8, label="not sig")
    ax.scatter(sig["effect_r"], -np.log10(sig["p_raw"]+1e-10),
               color="crimson", alpha=0.7, s=12, label=f"FDR sig (n={len(sig)})")
    ax.axhline(-np.log10(0.05), color="orange", linestyle="--",
               linewidth=1, label="p=0.05")
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_title(cond, fontsize=11)
    ax.set_xlabel("Effect size r")
    ax.set_ylabel("−log₁₀(p)")
    ax.legend(fontsize=7)

plt.tight_layout()
vol_path = os.path.join(OUTPUT_DIR, "fig_rq1_volcano_plots.png")
plt.savefig(vol_path, bbox_inches="tight"); plt.close()
print(f"\n  Saved: {vol_path}")

# ---- Figure 2: Top discriminative features bar chart ----
if len(top_feats_rq1) > 0:
    plot_df = top_feats_rq1.head(15).copy()
    plot_df["short_feat"] = (plot_df["feature"]
        .str.replace("__trial_mean","",regex=False)
        .str.replace("Linsole:","L:",regex=False)
        .str.replace("Rinsole:","R:",regex=False)
        .str[:45])
    plot_df = plot_df.sort_values("abs_r")

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ["crimson" if r > 0 else "steelblue"
              for r in plot_df["effect_r"]]
    ax.barh(plot_df["short_feat"], plot_df["abs_r"], color=colors, edgecolor="black", alpha=0.85)
    ax.axvline(0.3, color="orange", linestyle="--", linewidth=1.2, label="|r|=0.3 medium effect")
    ax.set_xlabel("Effect size |r| (rank-biserial correlation)")
    ax.set_title("RQ1 — Top 15 Discriminative Features (FDR-significant, ranked by |r|)",
                 fontweight="bold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    feat_bar_path = os.path.join(OUTPUT_DIR, "fig_rq1_top_features_effectsize.png")
    plt.savefig(feat_bar_path, bbox_inches="tight"); plt.close()
    print(f"  Saved: {feat_bar_path}")

# ---- Figure 3: Group comparison for top features ----
top6 = (stats_all[stats_all["significant"]]
        .assign(abs_r=lambda d: d["effect_r"].abs())
        .sort_values("abs_r", ascending=False)
        .drop_duplicates("feature")
        .head(6)["feature"].tolist())

if top6:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
    fig.suptitle("RQ1 — Group Comparison for Top 6 Discriminative Features",
                 fontweight="bold")
    for ai, feat in enumerate(top6):
        ax = axes[ai]
        feat_rows = stats_all[stats_all["feature"]==feat].sort_values("condition")
        conds_p = feat_rows["condition"].tolist()
        x = np.arange(len(conds_p)); w = 0.35
        ax.bar(x-w/2, feat_rows["HC_mean"], w, yerr=feat_rows["HC_std"],
               capsize=3, color=GROUP_PALETTE[0], alpha=0.8, label="HC")
        ax.bar(x+w/2, feat_rows["PD_mean"], w, yerr=feat_rows["PD_std"],
               capsize=3, color=GROUP_PALETTE[1], alpha=0.8, label="PD")
        for xi, (_, row) in enumerate(feat_rows.iterrows()):
            sig_str = ("***" if row["p_raw"]<0.001 else
                       "**"  if row["p_raw"]<0.01  else
                       "*"   if row["p_raw"]<0.05  else "ns")
            if row.get("significant", False):
                yp = max(row["PD_mean"]+row["PD_std"],
                         row["HC_mean"]+row["HC_std"])*1.08
                ax.text(xi+w/2, yp, sig_str, ha="center", fontsize=10)
        ax.set_xticks(x); ax.set_xticklabels(conds_p, rotation=20, ha="right", fontsize=8)
        short = feat.replace("__","—").replace("Linsole:","L:")[:50]
        ax.set_title(short, fontsize=8); ax.legend(fontsize=7)
    for ai in range(len(top6),6):
        axes[ai].axis("off")
    plt.tight_layout()
    grp_cmp_path = os.path.join(OUTPUT_DIR, "fig_rq1_group_comparisons.png")
    plt.savefig(grp_cmp_path, bbox_inches="tight"); plt.close()
    print(f"  Saved: {grp_cmp_path}")

## 7. RQ2 — Condition Sensitivity

Aggregate discriminability metrics per condition: number of FDR-significant features, mean |r|, feature-type breakdown.


In [ ]:

print("\n" + "="*68)
print("RQ2 — CONDITION SENSITIVITY")
print("="*68)

# Discriminability metrics per condition
rq2_rows = []
for cond in CONDITIONS:
    cdata = stats_all[stats_all["condition"] == cond].dropna(subset=["effect_r"])
    n_sig  = int(cdata["significant"].sum())
    n_tot  = len(cdata)
    pct_sig = 100 * n_sig / n_tot if n_tot > 0 else 0.0
    sig_r  = cdata.loc[cdata["significant"], "effect_r"].abs()
    mean_r = float(sig_r.mean()) if len(sig_r) > 0 else 0.0
    max_r  = float(sig_r.max())  if len(sig_r) > 0 else 0.0
    rq2_rows.append({
        "Condition":          cond,
        "N_features_tested":  n_tot,
        "N_significant_FDR":  n_sig,
        "Pct_significant":    round(pct_sig, 2),
        "Mean_abs_r_sig":     round(mean_r, 4),
        "Max_abs_r_sig":      round(max_r, 4),
    })

rq2_stat_df = pd.DataFrame(rq2_rows)
print("\nCondition discriminability summary (statistical):")
print(rq2_stat_df.to_string())
rq2_stat_csv = os.path.join(OUTPUT_DIR, "SPSS_rq2_condition_discriminability.csv")
rq2_stat_df.to_csv(rq2_stat_csv, index=False)

# Figure: significant features and mean effect size per condition
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("RQ2 — Condition Sensitivity: Statistical Discriminability",
             fontweight="bold")

sns.barplot(data=rq2_stat_df, x="Condition", y="N_significant_FDR",
            palette="viridis", ax=axes[0])
axes[0].set_title("Number of FDR-Significant Features")
axes[0].set_ylabel("Count")

sns.barplot(data=rq2_stat_df, x="Condition", y="Mean_abs_r_sig",
            palette="viridis", ax=axes[1])
axes[1].axhline(0.3, color="red", linestyle="--", linewidth=1.2,
                label="0.3 medium effect")
axes[1].set_title("Mean Effect Size |r| (Significant Features Only)")
axes[1].set_ylabel("|r| mean")
axes[1].legend(fontsize=9)

plt.tight_layout()
rq2_stat_path = os.path.join(OUTPUT_DIR, "fig_rq2_condition_stat_discriminability.png")
plt.savefig(rq2_stat_path, bbox_inches="tight"); plt.close()
print(f"\nSaved: {rq2_stat_path}")

# Feature-type breakdown per condition
# Classify each feature into: time / frequency / entropy / spatiotemporal
def feat_type(feat_name):
    fn = feat_name.lower()
    if any(k in fn for k in ["psd_","dominant_freq","spectral_entropy",
                               "total_power","spectral_centroid"]):
        return "frequency"
    if "perm_entropy" in fn:
        return "entropy"
    if any(k in fn for k in ["mean","std","cv","min","max","range",
                               "rms","energy","skewness","kurtosis",
                               "zcr","mcr"]):
        return "spatiotemporal"
    return "other"

sig_stats["feat_type"] = sig_stats["feature"].apply(feat_type)
type_cond = (sig_stats.groupby(["condition","feat_type"])
             .size().reset_index(name="count"))

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=type_cond, x="condition", y="count", hue="feat_type",
            palette="tab10", ax=ax)
ax.set_title("RQ2 — Significant Features by Type and Condition",
             fontweight="bold")
ax.set_ylabel("Number of significant features (FDR)")
ax.legend(title="Feature type")
plt.tight_layout()
type_path = os.path.join(OUTPUT_DIR, "fig_rq2_feature_type_breakdown.png")
plt.savefig(type_path, bbox_inches="tight"); plt.close()
print(f"Saved: {type_path}")

## 8. RQ3 — Model Validation (LOSO)

Leave-One-Subject-Out cross-validation for RBF-SVM, Random Forest, and MLP.  
Pooled (all conditions) and per-condition experiments.  
**Outputs:** ROC curves, Macro-F1 bar chart, Mannini sensitivity/specificity figure, confusion matrices, F1 variability boxplot.


In [ ]:

print("\n" + "="*68)
print("RQ3 — MODEL VALIDATION (LOSO)")
print("="*68)

def build_models():
    return {
        "RBF_SVM": Pipeline([
            ("imp", SimpleImputer(strategy="mean")),
            ("scl", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=10.0, gamma="scale",
                        class_weight="balanced", probability=True,
                        random_state=RANDOM_SEED)),
        ]),
        "RandomForest": Pipeline([
            ("imp", SimpleImputer(strategy="mean")),
            ("clf", RandomForestClassifier(n_estimators=200,
                        class_weight="balanced", random_state=RANDOM_SEED,
                        n_jobs=-1)),
        ]),
        "MLP": Pipeline([
            ("imp", SimpleImputer(strategy="mean")),
            ("scl", StandardScaler()),
            ("clf", MLPClassifier(hidden_layer_sizes=(128,64),
                        activation="relu", solver="adam", alpha=0.001,
                        learning_rate_init=0.001, learning_rate="adaptive",
                        max_iter=500, early_stopping=True,
                        validation_fraction=0.1,
                        random_state=RANDOM_SEED, verbose=False)),
        ]),
    }


def sens_spec(yt, yp):
    s  = recall_score(yt, yp, pos_label=1, zero_division=0)
    sp = recall_score(yt, yp, pos_label=0, zero_division=0)
    return float(s), float(sp)


def safe_auc(yt, yprob):
    if len(np.unique(yt)) < 2:
        return np.nan
    try:
        return float(roc_auc_score(yt, yprob))
    except Exception:
        return np.nan


def run_loso(X, y, groups, trials, exp_name="Exp"):
    X      = X.reset_index(drop=True)
    y      = np.asarray(y)
    groups = np.asarray(groups)
    trials = np.asarray(trials)

    if len(np.unique(y)) < 2:
        print(f"  SKIP {exp_name}: only one class.")
        return pd.DataFrame(), {}, []

    logo         = LeaveOneGroupOut()
    results, preds_dict, fold_records = [], {}, []

    for mn, model in build_models().items():
        print(f"\n  -- {mn} ({exp_name}) --")
        all_true, all_pred, all_prob = [], [], []
        all_trial, all_group = [], []

        for fi, (tr_i, te_i) in enumerate(logo.split(X, y, groups), 1):
            Xtr, Xte = X.iloc[tr_i], X.iloc[te_i]
            ytr, yte = y[tr_i], y[te_i]
            gte = groups[te_i]; tte = trials[te_i]
            subj = str(gte[0]); glbl = int(yte[0])

            if len(np.unique(ytr)) < 2:
                print(f"    Fold {fi:02d} [{subj}]: skip (1 class in train)")
                continue

            model.fit(Xtr, ytr)
            yp = model.predict(Xte)
            yprob = (model.predict_proba(Xte)[:,1]
                     if hasattr(model,"predict_proba") else yp.astype(float))

            fa = float(accuracy_score(yte, yp))
            ff = float(f1_score(yte, yp, average="macro", zero_division=0))
            fs, fsp = sens_spec(yte, yp)

            fold_records.append({
                "exp": exp_name, "model": mn, "fold": fi,
                "subject": subj, "group": glbl,
                "group_name": GROUP_LABELS[glbl],
                "accuracy": round(fa,4),
                "balanced_accuracy": round(float(balanced_accuracy_score(yte,yp)),4),
                "macro_f1": round(ff,4),
                "sensitivity": round(fs,4),
                "specificity": round(fsp,4),
                "pd_f1": round(float(f1_score(yte,yp,zero_division=0)),4),
                "n_test": int(len(np.unique(tte))),
            })
            print(f"    Fold {fi:02d} [{subj:18s}] "
                  f"acc={fa:.3f} F1={ff:.3f} "
                  f"sens={fs:.3f} spec={fsp:.3f}")

            all_true.extend(yte);  all_pred.extend(yp)
            all_prob.extend(yprob); all_trial.extend(tte)
            all_group.extend(gte)

        if not all_true:
            continue

        yt = np.array(all_true); yp = np.array(all_pred)
        yprob = np.array(all_prob)
        s, sp = sens_spec(yt, yp)
        auc   = safe_auc(yt, yprob)

        results.append({
            "Experiment":              exp_name,
            "Model":                   mn,
            "Accuracy":                round(float(accuracy_score(yt,yp)),4),
            "Balanced_Accuracy":       round(float(balanced_accuracy_score(yt,yp)),4),
            "Macro_F1":                round(float(f1_score(yt,yp,average="macro",zero_division=0)),4),
            "Sensitivity":             round(s,4),
            "Specificity":             round(sp,4),
            "PD_F1":                   round(float(f1_score(yt,yp,zero_division=0)),4),
            "Precision_PD":            round(float(precision_score(yt,yp,zero_division=0)),4),
            "AUC":                     round(auc,4) if auc and np.isfinite(auc) else np.nan,
        })
        preds_dict[mn] = {
            "y_true":yt,"y_pred":yp,"y_prob":yprob,
            "groups":np.array(all_group),"trial_ids":np.array(all_trial),
        }

    return pd.DataFrame(results), preds_dict, fold_records


# --- Run pooled experiment ---
print("\nPooled (all conditions):")
res_pool, preds_pool, folds_pool = run_loso(
    X_tr, y_tr, grp_tr, tri_tr, "Pooled")

# --- Run per-condition experiments ---
cond_results, cond_preds, cond_folds = [], {}, []
for cond in CONDITIONS:
    Xc = X_cond[cond]; yc = y_cond[cond]
    gc = grp_cond[cond]; tc = tri_cond[cond]
    print(f"\nCondition: {cond}  (trials={len(yc)}, "
          f"subjects={len(np.unique(gc))})")
    if len(np.unique(yc)) < 2 or len(np.unique(gc)) < 2:
        print(f"  Skipping {cond}: insufficient data.")
        continue
    rc, pc, fc = run_loso(Xc, yc, gc, tc, cond)
    cond_results.append(rc)
    cond_preds[cond] = pc
    for row in fc:
        row["condition"] = cond
    cond_folds.extend(fc)

all_cond_results = pd.concat(cond_results, ignore_index=True)
fold_df = pd.DataFrame(folds_pool)
cond_fold_df = pd.DataFrame(cond_folds)

print("\nPooled LOSO results:")
print(res_pool.to_string())
print("\nPer-condition LOSO results:")
print(all_cond_results.to_string())

# Mannini-style consolidated table
def n_by_group(grp_arr, y_arr):
    df = pd.DataFrame({"s":grp_arr,"l":y_arr}).drop_duplicates("s")
    return int((df["l"]==1).sum()), int((df["l"]==0).sum())

mann_rows = []
npd_p, nhc_p = n_by_group(grp_tr, y_tr)
for _, row in res_pool.iterrows():
    mann_rows.append({"Condition":"Pooled","Model":row["Model"],
        "N_PD":npd_p,"N_HC":nhc_p,
        "Sensitivity":row["Sensitivity"],"Specificity":row["Specificity"],
        "Accuracy":row["Accuracy"],"Macro_F1":row["Macro_F1"],"AUC":row["AUC"]})

for cond in CONDITIONS:
    m = tsk_tr == cond
    npd_c, nhc_c = n_by_group(grp_tr[m], y_tr[m])
    for _, row in all_cond_results[all_cond_results["Experiment"]==cond].iterrows():
        mann_rows.append({"Condition":cond,"Model":row["Model"],
            "N_PD":npd_c,"N_HC":nhc_c,
            "Sensitivity":row["Sensitivity"],"Specificity":row["Specificity"],
            "Accuracy":row["Accuracy"],"Macro_F1":row["Macro_F1"],"AUC":row["AUC"]})

mannini_df = pd.DataFrame(mann_rows)

# ---- Figures for RQ3 ----

# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("RQ3 — ROC Curves: Pooled LOSO Classification",
             fontweight="bold", fontsize=12)
for ax, (exp_name, preds) in zip(axes, [
    ("Pooled — all conditions", preds_pool),
    ("TUG — best single condition",
     cond_preds.get("TUG", {})),
]):
    for mn, p in preds.items():
        if len(np.unique(p["y_true"])) < 2:
            continue
        fpr, tpr, _ = roc_curve(p["y_true"], p["y_prob"])
        auc = roc_auc_score(p["y_true"], p["y_prob"])
        ax.plot(fpr, tpr, color=MODEL_COLORS[mn],
                label=f"{mn} (AUC={auc:.3f})", linewidth=2)
    ax.plot([0,1],[0,1],"k--",linewidth=1,label="Chance")
    ax.set_title(exp_name, fontsize=11)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
roc_path = os.path.join(OUTPUT_DIR, "fig_rq3_roc_curves.png")
plt.savefig(roc_path, bbox_inches="tight"); plt.close()
print(f"\nSaved: {roc_path}")

# Macro-F1 per condition bar chart
fig, ax = plt.subplots(figsize=(11, 5.5))
plot_df = all_cond_results.copy()
plot_df["Condition"] = pd.Categorical(
    plot_df["Experiment"], categories=CONDITIONS, ordered=True)
sns.barplot(data=plot_df, x="Condition", y="Macro_F1",
            hue="Model", hue_order=MODEL_ORDER,
            palette=MODEL_COLORS, ax=ax)
ax.axhline(0.70, color="red", linestyle="--", linewidth=1.2,
           label="0.70 clinical threshold")
ax.set_title("RQ3 — Macro-F1 by Walking Condition and Classifier",
             fontweight="bold")
ax.set_ylim(0,1); ax.legend(title="Model", loc="lower right")
plt.tight_layout()
f1_path = os.path.join(OUTPUT_DIR, "fig_rq3_condition_macro_f1.png")
plt.savefig(f1_path, bbox_inches="tight"); plt.close()
print(f"Saved: {f1_path}")

# Sensitivity/Specificity Mannini figure
all_conds_order = CONDITIONS + ["Pooled"]
mdf = mannini_df.copy()
mdf["Condition"] = pd.Categorical(
    mdf["Condition"], categories=all_conds_order, ordered=True)
mdf = mdf.sort_values("Condition")

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle("RQ3 — Sensitivity and Specificity by Condition and Classifier "
             "(Mannini et al. 2016 Framework)", fontweight="bold", fontsize=12)
for ax, metric, ylabel in zip(axes,
    ["Sensitivity","Specificity"],
    ["Sensitivity (PD Recall)","Specificity (HC Recall)"]):
    sns.barplot(data=mdf, x="Condition", y=metric,
                hue="Model", hue_order=MODEL_ORDER,
                palette=MODEL_COLORS, ax=ax)
    ax.axhline(0.70, color="red", linestyle="--", linewidth=1.2)
    ax.set_title(ylabel); ax.set_ylim(0,1)
    ax.legend(title="Model", loc="lower right", fontsize=8)
plt.tight_layout()
mann_path = os.path.join(OUTPUT_DIR, "fig_rq3_mannini_sens_spec.png")
plt.savefig(mann_path, bbox_inches="tight"); plt.close()
print(f"Saved: {mann_path}")

# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("RQ3 — Confusion Matrices, Pooled LOSO",
             fontweight="bold", fontsize=13)
for ci, mn in enumerate(MODEL_ORDER):
    if mn not in preds_pool:
        axes[0,ci].axis("off"); axes[1,ci].axis("off"); continue
    p = preds_pool[mn]
    for ri, (cmap, level) in enumerate([("Blues","Window"),("Greens","Trial")]):
        ConfusionMatrixDisplay(
            confusion_matrix(p["y_true"], p["y_pred"], labels=[0,1]),
            display_labels=["HC","PD"]
        ).plot(ax=axes[ri,ci], colorbar=False, cmap=cmap)
        axes[ri,ci].set_title(f"{mn} — {level}", fontsize=10)
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, "fig_rq3_confusion_matrices.png")
plt.savefig(cm_path, bbox_inches="tight"); plt.close()
print(f"Saved: {cm_path}")

# F1 variability boxplot
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.boxplot(data=fold_df, x="model", y="macro_f1",
            order=MODEL_ORDER, color="lightsteelblue", ax=ax)
sns.stripplot(data=fold_df, x="model", y="macro_f1",
              order=MODEL_ORDER, hue="group_name",
              dodge=True, size=5, alpha=0.75, ax=ax)
ax.set_title("RQ3 — Per-Subject Macro-F1 Variability, Pooled LOSO",
             fontweight="bold")
ax.set_ylim(0,1); ax.legend(title="Group", loc="lower right")
plt.tight_layout()
var_path = os.path.join(OUTPUT_DIR, "fig_rq3_f1_variability.png")
plt.savefig(var_path, bbox_inches="tight"); plt.close()
print(f"Saved: {var_path}")

## 9. RF Feature Importance

Fit Random Forest on the full trial-level dataset; plot top 20 features coloured by type (spatiotemporal / frequency / entropy).


In [ ]:

print("\nFitting RF on full pooled dataset for feature importance...")
rf_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="mean")),
    ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                    random_state=RANDOM_SEED, n_jobs=-1)),
])
rf_pipe.fit(X_tr, y_tr)
feat_imp = (pd.Series(rf_pipe.named_steps["clf"].feature_importances_,
                       index=X_tr.columns)
            .sort_values(ascending=False))
feat_imp_df = feat_imp.reset_index()
feat_imp_df.columns = ["feature","importance"]
feat_imp_df["feat_type"] = feat_imp_df["feature"].apply(feat_type)

fig, ax = plt.subplots(figsize=(11,7))
colors = [{"frequency":"coral","entropy":"gold","spatiotemporal":"steelblue"}
          .get(feat_type(f),"grey")
          for f in feat_imp.head(20).sort_values().index]
feat_imp.head(20).sort_values().plot(
    kind="barh", ax=ax, color=colors, edgecolor="black", alpha=0.85)
ax.set_title("RF Feature Importance — Top 20 (colour = feature type)",
             fontweight="bold")
ax.set_xlabel("Mean Decrease in Impurity")
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="steelblue", label="spatiotemporal"),
    Patch(color="coral",     label="frequency"),
    Patch(color="gold",      label="entropy"),
], loc="lower right")
plt.tight_layout()
rf_path = os.path.join(OUTPUT_DIR, "fig_rf_feature_importance.png")
plt.savefig(rf_path, bbox_inches="tight"); plt.close()
print(f"Saved: {rf_path}")

## 10. Statistical Tests for RQ3

Friedman test across models (pooled LOSO), post-hoc Wilcoxon pairwise.  
Wilcoxon signed-rank: pooled vs each single condition, per subject.


In [ ]:

print("\n" + "="*68)
print("RQ3 STATISTICAL TESTS")
print("="*68)

pivot_b = fold_df.pivot(index="subject", columns="model", values="macro_f1").dropna()
if len(pivot_b) >= 3 and set(MODEL_ORDER).issubset(pivot_b.columns):
    stat, pv = friedmanchisquare(
        pivot_b["RBF_SVM"], pivot_b["RandomForest"], pivot_b["MLP"])
    print(f"\nFriedman test (pooled, per-subject Macro-F1): "
          f"chi2={stat:.4f}, p={pv:.4f}")
    if pv < 0.05:
        print("  → Significant: post-hoc Wilcoxon pairwise:")
        for m1, m2 in itertools.combinations(MODEL_ORDER, 2):
            if m1 in pivot_b.columns and m2 in pivot_b.columns:
                try:
                    W, pw = wilcoxon(pivot_b[m1], pivot_b[m2])
                    sig = "* " if pw<0.05 else "ns"
                    print(f"    {m1} vs {m2}: W={W:.2f}, p={pw:.4f} [{sig}]")
                except Exception as e:
                    print(f"    {m1} vs {m2}: {e}")
    else:
        print("  → Non-significant: no pairwise follow-up needed.")

# Wilcoxon: pooled vs each condition
wilc_rows = []
pool_subj_f1 = {}
for mn in MODEL_ORDER:
    ps = fold_df[fold_df["model"]==mn].set_index("subject")["macro_f1"]
    pool_subj_f1[mn] = ps

print("\nWilcoxon signed-rank: pooled vs each condition (per-subject Macro-F1):")
for mn in MODEL_ORDER:
    for cond in CONDITIONS:
        sc = (cond_fold_df[(cond_fold_df["condition"]==cond)&
                            (cond_fold_df["model"]==mn)]
              .set_index("subject")["macro_f1"])
        paired = pd.concat([pool_subj_f1[mn].rename("p"),
                             sc.rename("s")], axis=1).dropna()
        if len(paired) >= 2:
            try:
                W, pw = wilcoxon(paired["p"], paired["s"])
            except Exception:
                W, pw = np.nan, np.nan
        else:
            W, pw = np.nan, np.nan
        sig = ("***" if pw<0.001 else "**" if pw<0.01 else
               "*" if pw<0.05 else "ns") if np.isfinite(pw) else "-"
        print(f"  {mn:14s} vs {cond:12s}: N={len(paired):3d} "
              f"W={W:.1f} p={pw:.4f} [{sig}]")
        wilc_rows.append({"Model":mn,"Condition":cond,
            "N_paired":len(paired),
            "Pooled_mean_F1":round(float(paired["p"].mean()),4) if len(paired) else np.nan,
            "Single_mean_F1":round(float(paired["s"].mean()),4) if len(paired) else np.nan,
            "Wilcoxon_W":round(float(W),4) if np.isfinite(W) else np.nan,
            "p_value":round(float(pw),4) if np.isfinite(pw) else np.nan,
            "sig":sig})

wilc_df = pd.DataFrame(wilc_rows)

## 11. Export Results

Save all result tables to `E:\DATASETT
esults_pdvshc_v3\` as SPSS-ready CSV files.


In [ ]:

print("\n" + "="*68)
print("EXPORTING RESULTS")
print("="*68)

exports = {
    "SPSS_rq1_feature_stats_all.csv":           stats_all,
    "SPSS_rq1_top_discriminative_features.csv":  top_feats_rq1,
    "SPSS_rq2_condition_discriminability.csv":   rq2_stat_df,
    "SPSS_rq3_pooled_results.csv":               res_pool,
    "SPSS_rq3_per_condition_results.csv":        all_cond_results,
    "SPSS_rq3_mannini_style.csv":                mannini_df,
    "SPSS_rq3_fold_metrics_long.csv":            fold_df,
    "SPSS_rq3_wilcoxon_pooled_vs_condition.csv": wilc_df,
    "SPSS_rq3_fold_macro_f1_wide.csv":           pivot_b.reset_index(),
    "SPSS_rf_feature_importances.csv":           feat_imp_df,
    "SPSS_rq2_significant_features.csv":         sig_stats,
}
for fname, df in exports.items():
    path = os.path.join(OUTPUT_DIR, fname)
    df.to_csv(path, index=False)
    print(f"  {fname:55s} ({len(df)} rows)")

## 12. Results Summary

Print concise RQ1/RQ2/RQ3 findings to console.


In [ ]:

print("\n" + "="*68)
print("RESULTS SUMMARY")
print("="*68)

print("\n--- RQ1: Feature Discriminability ---")
for cond in CONDITIONS + ["Pooled"]:
    row = rq2_stat_df[rq2_stat_df["Condition"]==cond]
    if not row.empty:
        r = row.iloc[0]
        print(f"  {cond:12s}: {int(r['N_significant_FDR']):4d} sig. features "
              f"| mean |r| = {r['Mean_abs_r_sig']:.3f} "
              f"| max |r| = {r['Max_abs_r_sig']:.3f}")
    else:
        cd = stats_all[stats_all["condition"]==cond]
        n = int(cd["significant"].sum()) if "significant" in cd.columns else 0
        print(f"  {cond:12s}: {n:4d} sig. features")

print("\n--- RQ2: Best Condition (statistical) ---")
best_cond_stat = rq2_stat_df.sort_values(
    ["N_significant_FDR","Mean_abs_r_sig"], ascending=False).iloc[0]
print(f"  Best: {best_cond_stat['Condition']} "
      f"({int(best_cond_stat['N_significant_FDR'])} sig. features, "
      f"mean |r| = {best_cond_stat['Mean_abs_r_sig']:.3f})")

print("\n--- RQ3: Best Model/Condition (LOSO) ---")
best_pool = (res_pool.sort_values("Macro_F1", ascending=False).iloc[0]
             if not res_pool.empty else None)
best_cond_cls = (all_cond_results.sort_values("Macro_F1", ascending=False).iloc[0]
                 if not all_cond_results.empty else None)
if best_pool is not None:
    print(f"  Best pooled : {best_pool['Model']} "
          f"F1={best_pool['Macro_F1']:.4f} "
          f"AUC={best_pool['AUC']:.4f} "
          f"Sens={best_pool['Sensitivity']:.4f} "
          f"Spec={best_pool['Specificity']:.4f}")
if best_cond_cls is not None:
    print(f"  Best single : {best_cond_cls['Experiment']} / {best_cond_cls['Model']} "
          f"F1={best_cond_cls['Macro_F1']:.4f} "
          f"AUC={best_cond_cls['AUC']:.4f}")

## 13. Artefact Manifest

List all CSV and PNG outputs with file sizes.


In [ ]:

print("\n" + "="*68)
print("FINAL ARTEFACT MANIFEST")
print("="*68)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    if fname.lower().endswith((".csv",".png")):
        kb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
        print(f"  {fname:60s}  {kb:8.1f} KB")
print("\nPipeline v3 complete.")